# BRFSS 2025 Diabetes Classification — End-to-End Study

**Research objective:** compare machine-learning algorithms for classifying **self-reported diagnosed diabetes status** using demographic, behavioral, socioeconomic, healthcare-access, and health indicators from the 2025 Behavioral Risk Factor Surveillance System (BRFSS).

## Canonical notebook rule

This is the **single executable source of truth** for the BRFSS 2025 study. It contains the complete research flow in one sequential notebook:

1. dataset choice, provenance, and limitations;
2. literature-grounded target and predictor selection;
3. analytical recoding and missing-data handling;
4. leakage-safe train/test split and preprocessing;
5. Models 01–07 under the same frozen protocol;
6. per-model CV, held-out test metrics, bootstrap uncertainty, and diagnostics;
7. consolidated seven-model benchmark;
8. respondent-level paired statistical comparison;
9. final interpretation and Part III synthesis.

The notebook deliberately reuses the same cohort, split, preprocessing contract, fitted model outputs, and held-out predictions across later sections. **No separate model or comparison notebook is required.**


# I. BRFSS 2025 Dataset and Prior Research

## 1.1 Why BRFSS for Diabetes Classification?

The Behavioral Risk Factor Surveillance System (BRFSS) is a large U.S. health survey coordinated by the Centers for Disease Control and Prevention (CDC). It collects information on health conditions, health-related behaviors, healthcare access, and demographic characteristics.

BRFSS 2025 is suitable for this study because it provides:

- a large population-level sample;
- a diabetes-status question (`DIABETE4`);
- demographic, behavioral, socioeconomic, and chronic-condition indicators in the same survey;
- publicly documented variable definitions and coding;
- a recent 2025 snapshot for an up-to-date comparative ML study.

### Important limitation

The diabetes outcome is **self-reported diagnosed diabetes status**. It is not a laboratory diagnosis, not a screen for undiagnosed diabetes, and not a longitudinal prediction of future diabetes.

**Official data source:** CDC BRFSS 2025 annual data.



## 1.2 Prior BRFSS Diabetes Machine-Learning Research

Our design is grounded in prior BRFSS-based diabetes studies rather than selecting variables or algorithms only by intuition.

Key anchors include:

1. **Nayem & Biswas (2026), Scientific Reports** — raw BRFSS 2014–2024, large-scale logistic modeling, literature-grounded predictor selection, public R code. DOI: `10.1038/s41598-026-46927-7`.
2. **Xie et al. (2019), Preventing Chronic Disease / CDC** — raw BRFSS 2014, literature-derived diabetes predictors, multiple ML models, leakage-safe train-only SMOTE. DOI: `10.5888/pcd16.190109`.
3. **Muhammad et al. (2025), Journal of Primary Care & Community Health** — BRFSS 2023 diabetes classification with seven ML algorithms, stratified split, 5-fold CV, imbalance-aware metrics, and SHAP. DOI: `10.1177/21501319251400546`.
4. **Aich et al. (2026), Scientific Reports** — BRFSS-derived diabetes health indicators with public Python code and reproducible feature-selection experiments. DOI: `10.1038/s41598-026-41874-9`.

These studies provide precedents for target construction, candidate-variable domains, evaluation design, feature selection, and class-imbalance handling.

**Reproducibility note:** literature is treated as evidence, not as unquestioned ground truth. Where published text and public code disagree, the discrepancy is documented and independently checked.



# II. Initial Exploration of BRFSS 2025

## 2.1 Data Source and File Integrity

The source file used here is the official CDC 2025 SAS Transport dataset (`.XPT`). The GitHub/Kaggle pipeline verifies the payload before analysis.

The original CDC ZIP is preserved separately. Kaggle uses a normalized archive member name because the CDC ZIP contains an XPT filename with a trailing space.



In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import pandas as pd
from IPython.display import display

TARGET = "DIABETE4"
EXPECTED_XPT_BYTES = 799_971_280

xpt_candidates = list(Path("/kaggle/input").rglob("LLCP2025.XPT"))
zip_candidates = list(Path("/kaggle/input").rglob("LLCP2025XPT.zip"))

print("XPT candidates:", [str(p) for p in xpt_candidates])
print("ZIP candidates:", [str(p) for p in zip_candidates])

if len(xpt_candidates) == 1:
    xpt_path = xpt_candidates[0]
    source_mode = "Kaggle dataset extracted XPT"
elif len(xpt_candidates) == 0 and len(zip_candidates) == 1:
    zip_path = zip_candidates[0]
    working_dir = Path("/kaggle/working/brfss_2025")
    working_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(zip_path) as zf:
        members = [info for info in zf.infolist() if not info.is_dir()]
        if len(members) != 1:
            raise RuntimeError(f"Expected one data payload, found {len(members)}.")
        member = members[0]
        normalized_name = Path(member.filename.strip()).name
        if normalized_name != "LLCP2025.XPT":
            raise RuntimeError(f"Unexpected payload name: {normalized_name!r}")
        xpt_path = working_dir / normalized_name
        with zf.open(member) as src, xpt_path.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                dst.write(chunk)
    source_mode = "Kaggle dataset ZIP fallback"
else:
    raise RuntimeError(
        "Expected exactly one BRFSS input payload from Kaggle Dataset. "
        f"Found {len(xpt_candidates)} XPT and {len(zip_candidates)} ZIP candidates."
    )

actual_bytes = xpt_path.stat().st_size
print("Source mode:", source_mode)
print("XPT path:", xpt_path)
print("XPT bytes:", f"{actual_bytes:,}")

if actual_bytes != EXPECTED_XPT_BYTES:
    raise RuntimeError(
        f"Unexpected XPT size: {actual_bytes:,}; expected {EXPECTED_XPT_BYTES:,}."
    )

sha256 = hashlib.sha256()
with xpt_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)
xpt_sha256 = sha256.hexdigest()
print("XPT SHA-256:", xpt_sha256)


In [ ]:
print("Validated Kaggle input payload.")
print("Ready to load SAS Transport file:", xpt_path)


## 2.2 Load and Inspect the Dataset

We first verify the shape, memory footprint, target availability, and a small sample of rows before performing any analytical recoding.



In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

print("Shape:", df.shape)
print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])
print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**3):.3f} GiB")

if TARGET not in df.columns:
    raise RuntimeError(f"Target {TARGET!r} not found in dataset.")

display(df.head())


## 2.3 Target Variable — `DIABETE4`

`DIABETE4` is the candidate outcome variable. We inspect its raw response distribution **before** recoding.

Primary study definition, frozen after literature and coding review:

- `1` → Diabetes
- `3` → No diabetes
- `2` → exclude (pregnancy-only diabetes)
- `4` → exclude (prediabetes / borderline)
- `7` → exclude (don't know / not sure)
- `9` → exclude (refused)
- missing → exclude

This creates a conservative binary comparison of diagnosed diabetes versus no diagnosed diabetes.



In [ ]:
target_counts = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="count")
)
target_counts["percent"] = target_counts["count"] / len(df) * 100

display(target_counts)
target_counts.to_csv("/kaggle/working/target_distribution.csv", index=False)

primary_target = df[TARGET].isin([1.0, 3.0])
n_primary = int(primary_target.sum())
n_diabetes = int((df[TARGET] == 1.0).sum())
n_no_diabetes = int((df[TARGET] == 3.0).sum())

print(f"Primary binary cohort: {n_primary:,}")
print(f"Diabetes: {n_diabetes:,}")
print(f"No diabetes: {n_no_diabetes:,}")
print(f"Diabetes prevalence in binary cohort: {n_diabetes / n_primary * 100:.2f}%")


## 2.4 Missingness and Variable Coverage

BRFSS uses two kinds of missingness:

1. native SAS missing values, which appear as `NaN`;
2. explicit response codes such as `7`, `9`, `77`, or `99` for don't know/refused/not sure, depending on the variable.

Therefore, raw `NaN` percentages are useful for inspection but are **not yet analytical missingness**. Coding-specific missing values are handled in the variable coding audit before preprocessing.



In [ ]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100),
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

print("Columns with the highest native SAS missingness:")
display(column_summary.head(30))

column_summary.to_csv("/kaggle/working/column_summary.csv", index=False)


# III. Literature-Grounded Variable Selection

## 3.1 Selection Criteria

A variable is retained only when it satisfies the study design rather than simply showing a strong univariate association with the target.

The selection criteria are:

1. **Literature support** — previously used or justified in BRFSS diabetes research.
2. **Analytical coverage** — acceptable missingness after decoding BRFSS special values.
3. **No target leakage** — the variable must not reveal treatment, monitoring, or history conditional on an existing diabetes diagnosis.
4. **Limited redundancy** — avoid duplicate representations of the same construct.
5. **Interpretability and domain coverage** — retain a compact set covering major health and social domains.



## 3.2 Explicit Target-Leakage Exclusions

The following variables are **not permitted** as predictors:

- `DIABAGE4` — age at diabetes diagnosis;
- `INSULIN1` — insulin use;
- `CHKHEMO3` — diabetes-specific HbA1c monitoring;
- `EYEEXAM1` — diabetes-specific eye examination;
- `DIABEYE1` — diabetes-related eye disease;
- `DIABEDU1` — diabetes self-management education.

These variables are downstream of, or conditional on, a known diabetes diagnosis. Their exclusion is based on leakage risk, not merely their high missingness.



## 3.3 Final Primary Predictor Set

The primary predictor set is frozen at **20 variables**.

It intentionally covers demographics, socioeconomic status, anthropometrics, general/mental health, lifestyle, cardiometabolic health, major comorbidities, functional health, and healthcare access while avoiding unnecessary duplication.



In [ ]:
PRIMARY_FEATURES = [
    "_AGEG5YR",
    "_SEX",
    "_RACEGR3",
    "_EDUCAG",
    "_INCOMG1",
    "EMPLOY1",
    "_BMI5",
    "GENHLTH",
    "PHYSHLTH",
    "MENTHLTH",
    "_TOTINDA",
    "_RFSMOK3",
    "SLEPTIM1",
    "_RFHYPE6",
    "_RFCHOL3",
    "_MICHD",
    "CVDSTRK3",
    "CHCKDNY2",
    "DIFFWALK",
    "PERSDOC3"
]

feature_domains = {
    "_AGEG5YR": "Demographic",
    "_SEX": "Demographic",
    "_RACEGR3": "Demographic",
    "_EDUCAG": "Socioeconomic",
    "_INCOMG1": "Socioeconomic",
    "EMPLOY1": "Socioeconomic",
    "_BMI5": "Anthropometric",
    "GENHLTH": "General health",
    "PHYSHLTH": "General health",
    "MENTHLTH": "Mental health",
    "_TOTINDA": "Lifestyle",
    "_RFSMOK3": "Lifestyle",
    "SLEPTIM1": "Lifestyle",
    "_RFHYPE6": "Cardiometabolic",
    "_RFCHOL3": "Cardiometabolic",
    "_MICHD": "Comorbidity",
    "CVDSTRK3": "Comorbidity",
    "CHCKDNY2": "Comorbidity",
    "DIFFWALK": "Functional health",
    "PERSDOC3": "Healthcare access",
}

missing_features = [c for c in PRIMARY_FEATURES if c not in df.columns]
if missing_features:
    raise RuntimeError(f"Frozen predictors missing from BRFSS 2025: {missing_features}")

feature_audit = pd.DataFrame({
    "variable": PRIMARY_FEATURES,
    "domain": [feature_domains[c] for c in PRIMARY_FEATURES],
    "native_missing_pct": [df[c].isna().mean() * 100 for c in PRIMARY_FEATURES],
    "n_unique_nonnull": [df[c].nunique(dropna=True) for c in PRIMARY_FEATURES],
})

display(feature_audit)
feature_audit.to_csv("/kaggle/working/final_feature_audit.csv", index=False)


## 3.4 Why Some Plausible Variables Were Not Selected

Examples of deliberate exclusions:

- `_BMI5CAT` — redundant with continuous `_BMI5`.
- `_RFHLTH` — derived from `GENHLTH`.
- `MARITAL` — lower priority after age, sex, education, income, employment, and race/ethnicity are represented.
- `_HLTHPL2`, `MEDCOST1`, `CHECKUP1` — useful access/use variables, but only `PERSDOC3` is retained in the compact primary set.
- `DRNKANY6`, `_RFBING6` — valid lifestyle variables, but activity, smoking, and sleep already provide interpretable behavioral coverage.
- `ASTHMA3`, `HAVARTH4` — weaker diabetes-specific rationale than the selected cardiovascular, kidney, and functional-health indicators.
- `ADDEPEV3` — overlaps the broader `MENTHLTH` construct.

These variables remain available for sensitivity analyses but are not part of the primary benchmark.



## 3.5 Interpretation Note

`_RACEGR3` is retained as a population-demographic covariate because prior raw-BRFSS studies include race/ethnicity and because diabetes prevalence can differ across population groups.

Any association must **not** be interpreted as a biological causal effect. Observed differences may reflect social, environmental, healthcare-access, socioeconomic, and structural factors not fully represented by the remaining variables.



# IV. Part I Summary

At the end of Part I we have:

- verified and loaded the official BRFSS 2025 public-use file;
- inspected the raw dataset structure and missingness;
- defined a conservative binary `DIABETE4` target;
- grounded predictor selection in prior BRFSS diabetes literature;
- screened diabetes-specific target leakage;
- frozen a compact **20-predictor** primary feature set.

The next section addresses **analytical recoding, missing-data handling, encoding, train/test separation, and cross-validation**. None of those transformations is fit using information from the final test set.



In [ ]:
overview = {
    "source_mode": source_mode,
    "source_file": xpt_path.name,
    "xpt_sha256": xpt_sha256,
    "xpt_bytes": int(actual_bytes),
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "target": TARGET,
    "target_missing": int(df[TARGET].isna().sum()),
    "memory_gib": round(float(df.memory_usage(deep=True).sum() / (1024**3)), 3),
}

with open("/kaggle/working/dataset_overview.json", "w", encoding="utf-8") as f:
    json.dump(overview, f, indent=2)

print(json.dumps(overview, indent=2))
print("Initial exploration complete.")


# Part II — Data Preparation

## 2.1 Preprocessing Principles

Part II converts the raw BRFSS responses into a model-ready dataset while preventing data leakage.

The rules are:

1. construct the binary target first and exclude non-target response categories;
2. decode BRFSS special values such as `7`, `9`, `77`, and `99` before measuring analytical missingness;
3. split train/test **before fitting any imputer, encoder, or scaler**;
4. fit preprocessing only on the training set;
5. keep the test set untouched until final model evaluation;
6. perform 5-fold cross-validation only within the training set.

This project is a **machine-learning classification comparison**, not a design-based BRFSS prevalence analysis. Survey weights such as `_LLCPWT` are therefore not used as predictors or for the primary model benchmark. Any population prevalence estimate would require the BRFSS complex-sampling design and weights.



## 2.2 Construct the Binary Target and Recode the 20 Predictors

The primary outcome compares respondents reporting diagnosed diabetes (`DIABETE4=1`) with respondents reporting no diabetes (`DIABETE4=3`).

Gestational diabetes, prediabetes/borderline diabetes, don't-know/refused responses, and missing target responses are excluded from the primary binary cohort.

For predictors, CDC non-response codes are converted to analytical missing values. Examples:

- `_AGEG5YR=14` → missing;
- `_EDUCAG=9` → missing;
- `_INCOMG1=9` → missing;
- `PHYSHLTH/MENTHLTH=88` → zero days, while 77/99 → missing;
- `SLEPTIM1=1..24` → valid hours, while 77/99 → missing;
- calculated binary variables with code 9 → missing.

The continuous BMI field `_BMI5` is stored as BMI × 100 and is divided by 100.



In [ ]:
import numpy as np

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)

    # Categorical / ordinal survey variables
    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    # Continuous variables
    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)

    return out[PRIMARY_FEATURES]

target_mask = df["DIABETE4"].isin([1.0, 3.0])
analysis_raw = df.loc[target_mask, PRIMARY_FEATURES].copy()
X = recode_brfss_2025(analysis_raw)
y = (df.loc[target_mask, "DIABETE4"] == 1.0).astype("int8").rename("diabetes")

assert len(X) == len(y)
assert X.columns.tolist() == PRIMARY_FEATURES
assert set(y.unique()) == {0, 1}

print("Analysis cohort:", f"{len(X):,}")
print("Diabetes:", f"{int(y.sum()):,}")
print("No diabetes:", f"{int((y == 0).sum()):,}")
print("Sample diabetes prevalence:", f"{y.mean() * 100:.2f}%")
display(X.head())


## 2.3 Analytical Missingness After Recoding

Analytical missingness is recomputed **after** special-value decoding. This is the missingness relevant to preprocessing.

We also quantify how many respondents would remain under a strict complete-case strategy. This provides an empirical reason for choosing imputation rather than automatically discarding every row with at least one missing predictor.



In [ ]:
analytical_missingness = pd.DataFrame({
    "variable": PRIMARY_FEATURES,
    "domain": [feature_domains[c] for c in PRIMARY_FEATURES],
    "missing_count": [int(X[c].isna().sum()) for c in PRIMARY_FEATURES],
    "missing_pct": [float(X[c].isna().mean() * 100) for c in PRIMARY_FEATURES],
    "n_unique_nonnull": [int(X[c].nunique(dropna=True)) for c in PRIMARY_FEATURES],
}).sort_values("missing_pct", ascending=False)

complete_case_mask = X.notna().all(axis=1)
complete_case_n = int(complete_case_mask.sum())
complete_case_pct = complete_case_n / len(X) * 100

print(f"Rows in primary cohort: {len(X):,}")
print(f"Complete cases across all 20 predictors: {complete_case_n:,} ({complete_case_pct:.2f}%)")
print(f"Rows lost by complete-case deletion: {len(X)-complete_case_n:,} ({100-complete_case_pct:.2f}%)")

display(analytical_missingness)
analytical_missingness.to_csv("/kaggle/working/analytical_missingness.csv", index=False)


## 2.4 Missing-Data Strategy

The primary pipeline **retains the full binary target cohort** and handles missing predictors inside the training pipeline:

- **Categorical predictors:** missing values become an explicit `-1` category, then one-hot encoded.
- **Continuous predictors:** missing values are median-imputed using the training data only.
- **Continuous scaling:** standardized using training-set mean and standard deviation.

Why not complete-case deletion?

- income alone has substantial non-response;
- BMI, cholesterol, smoking, and sleep also contribute missingness;
- deleting every row with any missing predictor can remove a large fraction of respondents and alter the analyzed sample.

Why an explicit category for categorical missingness?

It avoids inventing a substantive category such as the modal income group for a respondent who refused or did not report income. This choice is transparent and can later be tested in sensitivity analysis.



## 2.5 Train/Test Split

We use an **80/20 stratified split** with a fixed random seed.

- 80% training set: used for preprocessing fitting, cross-validation, model selection, and tuning.
- 20% test set: untouched until final evaluation.
- stratification preserves the diabetes/non-diabetes class ratio.

This split is respondent-level because the public BRFSS file contains cross-sectional survey records rather than longitudinal patient trajectories.



In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame([
    {
        "split": "full",
        "n": len(y),
        "diabetes_n": int(y.sum()),
        "no_diabetes_n": int((y == 0).sum()),
        "diabetes_pct": float(y.mean() * 100),
    },
    {
        "split": "train",
        "n": len(y_train),
        "diabetes_n": int(y_train.sum()),
        "no_diabetes_n": int((y_train == 0).sum()),
        "diabetes_pct": float(y_train.mean() * 100),
    },
    {
        "split": "test",
        "n": len(y_test),
        "diabetes_n": int(y_test.sum()),
        "no_diabetes_n": int((y_test == 0).sum()),
        "diabetes_pct": float(y_test.mean() * 100),
    },
])

display(split_summary)
split_summary.to_csv("/kaggle/working/split_summary.csv", index=False)

assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train) + len(X_test) == len(X)


## 2.6 Leakage-Safe Preprocessing Pipeline

A single reusable `ColumnTransformer` is defined for the baseline model comparison.

Categorical variables are imputed with an explicit missing category and one-hot encoded. Numerical variables are median-imputed and standardized.

The transformer is **fit on `X_train` only**. Calling `fit_transform` on the full dataset before the split would leak information from the test set.



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Training matrix:", X_train_ready.shape)
print("Test matrix:", X_test_ready.shape)
print("Transformed feature count:", len(feature_names))
print("First transformed feature names:")
print(feature_names[:25])

assert X_train_ready.shape[0] == len(X_train)
assert X_test_ready.shape[0] == len(X_test)
assert X_train_ready.shape[1] == X_test_ready.shape[1]


## 2.7 Stratified 5-Fold Cross-Validation

Model comparison will use **Stratified 5-Fold CV inside the training set**.

The preprocessing transformer will later be placed inside each model pipeline, so every CV fold fits its own imputation, encoding, and scaling using only that fold's training partition.

This prevents validation-fold information from influencing preprocessing.



In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_rows = []
for fold, (fit_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    y_fit = y_train.iloc[fit_idx]
    y_val = y_train.iloc[val_idx]
    fold_rows.append({
        "fold": fold,
        "fit_n": len(fit_idx),
        "validation_n": len(val_idx),
        "fit_diabetes_pct": float(y_fit.mean() * 100),
        "validation_diabetes_pct": float(y_val.mean() * 100),
    })

cv_fold_summary = pd.DataFrame(fold_rows)
display(cv_fold_summary)
cv_fold_summary.to_csv("/kaggle/working/cv_fold_summary.csv", index=False)


## 2.8 Frozen Data-Preparation Design

The following decisions are now frozen for the primary benchmark:

- **Target:** `DIABETE4=1` vs `DIABETE4=3`.
- **Predictors:** the 20 literature-grounded variables from Part I.
- **Categorical missingness:** explicit missing category.
- **Continuous missingness:** median imputation.
- **Encoding:** one-hot encoding with unknown-category handling.
- **Scaling:** standardize continuous predictors.
- **Split:** 80/20 stratified holdout, `random_state=42`.
- **Cross-validation:** stratified 5-fold CV within training data only.
- **Resampling:** none in the primary baseline. Any SMOTE/undersampling experiment must occur inside training/CV folds only.
- **Test set:** untouched until the final selected models are evaluated.

Part III can therefore compare algorithms on the **same cohort, same split, and same preprocessing logic**, making differences in performance attributable primarily to the algorithms rather than to inconsistent data preparation.



In [ ]:
preprocessing_summary = {
    "target_definition": "DIABETE4 1 vs 3",
    "n_primary_cohort": int(len(y)),
    "n_diabetes": int(y.sum()),
    "sample_diabetes_pct": round(float(y.mean() * 100), 4),
    "n_predictors_raw": len(PRIMARY_FEATURES),
    "complete_case_n": complete_case_n,
    "complete_case_pct": round(float(complete_case_pct), 4),
    "train_n": int(len(y_train)),
    "test_n": int(len(y_test)),
    "train_diabetes_pct": round(float(y_train.mean() * 100), 4),
    "test_diabetes_pct": round(float(y_test.mean() * 100), 4),
    "transformed_feature_count": int(len(feature_names)),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "cv_folds": 5,
    "categorical_missing_strategy": "constant -1 category",
    "numeric_missing_strategy": "median fit on training only",
    "categorical_encoding": "one-hot, handle_unknown=ignore",
    "numeric_scaling": "StandardScaler fit on training only",
}

with open("/kaggle/working/preprocessing_summary.json", "w", encoding="utf-8") as f:
    json.dump(preprocessing_summary, f, indent=2)

print(json.dumps(preprocessing_summary, indent=2))


# Part III — Model Development and Evaluation

## 3.1 Model 01 — Logistic Regression

### Why Logistic Regression is first

Logistic Regression is used as the **baseline linear classifier**.

This choice is grounded in prior BRFSS diabetes studies:

- **Xie et al. (2019), Preventing Chronic Disease / CDC** benchmarked Logistic Regression alongside SVM, Decision Tree, Random Forest, Naive Bayes, and Neural Network on BRFSS 2014. DOI: `10.5888/pcd16.190109`.
- **Muhammad et al. (2025), Journal of Primary Care & Community Health** used Logistic Regression as the baseline linear model in a seven-model BRFSS 2023 diabetes comparison with stratified 5-fold CV. DOI: `10.1177/21501319251400546`.
- **Nayem & Biswas (2026), Scientific Reports** used logistic regression directly on large-scale BRFSS 2014–2024 and released public R code. DOI: `10.1038/s41598-026-46927-7`.

### Architecture used here

The predictive form is:

[
P(Y=1 mid X) = sigma(eta_0 + Xeta)
]

where (sigma(z)=1/(1+e^{-z})).

Our implementation is a **scikit-learn L2-regularized Logistic Regression** placed after the frozen Part II preprocessing pipeline.

Fixed configuration:

```text
solver       = lbfgs
penalty      = L2
C            = 1.0
class_weight = None
max_iter     = 1000
threshold    = 0.5
```

### What comes from the papers vs. what is ours

**Literature-supported**
- Logistic Regression as a BRFSS diabetes baseline;
- stratified holdout evaluation;
- 5-fold cross-validation;
- reporting discrimination and class-specific performance.

**Study-specific implementation choice**
- scikit-learn's L2-regularized implementation;
- fixed `C=1.0` with no hyperparameter tuning for this baseline;
- no SMOTE in the primary benchmark;
- threshold fixed at 0.5.

The Tennessee BRFSS paper used SMOTE-balanced training data. We deliberately do **not** copy that choice into the primary baseline because Part II froze the natural class distribution. Any resampling experiment will be handled separately inside training/CV folds only.

This is therefore a **literature-grounded implementation, not an exact reproduction** of any single prior paper.


### 3.1.1 Evaluation Protocol

The model uses exactly the same frozen data protocol defined in Part II:

- training set: 274,031 respondents;
- internal test set: 68,508 respondents;
- preprocessing inside the model pipeline;
- Stratified 5-Fold CV on the training set;
- untouched test set evaluated after the baseline configuration is fixed;
- no resampling;
- probability threshold = 0.5.

Metrics:

- Accuracy
- Precision
- Recall / Sensitivity
- Specificity
- F1
- ROC-AUC
- PR-AUC / Average Precision
- Brier score
- Confusion matrix

Cross-validation metrics are summarized with mean, standard deviation, and a 95% t-interval across the five folds. Final-test uncertainty is estimated with a stratified bootstrap.


In [ ]:
from pathlib import Path
from math import sqrt
import platform
import sklearn
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    make_scorer,
)
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

MODEL_01_DIR = Path("/kaggle/working/model_01_logistic_regression")
MODEL_01_DIR.mkdir(parents=True, exist_ok=True)

logreg = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    class_weight=None,
)

logreg_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", logreg),
])

specificity_scorer = make_scorer(recall_score, pos_label=0)
precision_scorer = make_scorer(precision_score, zero_division=0)

scoring = {
    "accuracy": "accuracy",
    "precision": precision_scorer,
    "recall": "recall",
    "specificity": specificity_scorer,
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "LogisticRegression",
    "solver": "lbfgs",
    "penalty": "l2",
    "C": 1.0,
    "max_iter": 1000,
    "class_weight": None,
    "threshold": 0.5,
    "random_state_split": RANDOM_STATE,
    "cv_folds": 5,
}

with open(MODEL_01_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)

print(json.dumps(environment, indent=2))


### 3.1.2 Five-Fold Cross-Validation on the Training Set

The entire model pipeline is passed into cross-validation. Therefore each fold independently fits:

1. categorical missing-value handling;
2. one-hot encoding;
3. numeric median imputation;
4. numeric scaling;
5. Logistic Regression.

The validation fold is never used to fit preprocessing statistics.


In [ ]:
cv_raw = cross_validate(
    logreg_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_records = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(cv_raw["score_time"][fold])
    fold_records.append(row)

logreg_cv_folds = pd.DataFrame(fold_records)

summary_records = []
for metric in metric_order:
    values = logreg_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    se = sd / sqrt(len(values))
    margin = t_critical_df4_95 * se
    summary_records.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

logreg_cv_summary = pd.DataFrame(summary_records)

display(logreg_cv_folds)
display(logreg_cv_summary)

logreg_cv_folds.to_csv(MODEL_01_DIR / "cv_folds.csv", index=False)
logreg_cv_summary.to_csv(MODEL_01_DIR / "cv_summary.csv", index=False)


### 3.1.3 Fit the Fixed Baseline on the Full Training Set

After cross-validation, the same fixed configuration is fit once on the complete training partition. No CV or test metric is used to change its hyperparameters.

The internal test set is then evaluated once at the standard probability threshold of 0.5.


In [ ]:
logreg_pipeline.fit(X_train, y_train)

y_test_prob = logreg_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred, labels=[0, 1]).ravel()

test_point_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
    "precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob)),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "threshold": 0.5,
}

print(json.dumps(test_point_metrics, indent=2))


### 3.1.4 Stratified Bootstrap Confidence Intervals

To quantify uncertainty on the held-out test set, we bootstrap diabetic and non-diabetic respondents separately with replacement. This preserves the test-set class composition while estimating sampling variability.

The test set is **not** used for parameter tuning; bootstrap samples are used only to estimate uncertainty around the already-fixed model's performance.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251003

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()

    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)

test_summary_rows = []
for metric in ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc", "brier_score"]:
    values = bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": test_point_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

logreg_test_summary = pd.DataFrame(test_summary_rows)

display(logreg_test_summary)

bootstrap_df.to_csv(MODEL_01_DIR / "bootstrap_metrics.csv", index=False)
logreg_test_summary.to_csv(MODEL_01_DIR / "test_summary.csv", index=False)

with open(MODEL_01_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_point_metrics, f, indent=2)


### 3.1.5 Diagnostic Curves and Confusion Matrix

ROC and precision-recall curves are saved for comparison with later models. The confusion matrix uses the fixed 0.5 threshold.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob)

fig = plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC={test_point_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 01 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_01_DIR / "roc_curve.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(6, 5))
plt.plot(recall_curve, precision_curve, label=f"Logistic Regression (AP={test_point_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 01 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_01_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp], [fn, tp]])
fig = plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Model 01 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0, 1], ["No diabetes", "Diabetes"])
plt.yticks([0, 1], ["No diabetes", "Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_01_DIR / "confusion_matrix.png", dpi=160)
plt.show()


### 3.1.6 Model 01 Freeze Point

After this run, Model 01 is considered frozen as:

> **L2 Logistic Regression, C=1.0, lbfgs, no class weighting, no resampling, threshold=0.5**

The model is not retuned after seeing the internal test results.

Its role in the study is the transparent linear baseline against which non-linear and ensemble models will be compared.

The next model should be implemented only after this baseline's execution artifacts and results have been reviewed.


## 3.2 Model 02 — Decision Tree

### Why Decision Tree is included

Decision Tree provides a transparent **non-linear, tree-based baseline**.

Prior BRFSS diabetes studies provide direct precedent:

- **Xie et al. (2019), Preventing Chronic Disease / CDC** evaluated a Decision Tree on BRFSS 2014. DOI: `10.5888/pcd16.190109`.
- **Muhammad et al. (2025), Journal of Primary Care & Community Health** included Decision Tree among seven BRFSS 2023 classifiers and described it as a tree-based model for capturing hierarchical decision patterns. DOI: `10.1177/21501319251400546`.

The Tennessee study states that default hyperparameters were used initially and tuning was performed where appropriate. Its published main text does not provide enough verified detail to claim that our exact tree configuration reproduces its tuned Decision Tree.

### Architecture used here

Scikit-learn uses an optimized CART-style algorithm that recursively partitions the feature space using binary splits.

This study uses a fixed **Gini-impurity Decision Tree**:

```text
criterion             = gini
splitter              = best
max_depth             = None
min_samples_split     = 2
min_samples_leaf      = 1
max_features          = None
class_weight          = None
ccp_alpha             = 0.0
random_state          = 42
threshold             = 0.5
```

This is intentionally an **unpruned default-style tree baseline**.

### Provenance

**Literature-supported**
- Decision Tree as a BRFSS diabetes classifier;
- tree-based non-linear modeling role;
- stratified train/test evaluation;
- 5-fold CV;
- standard classification metrics.

**Study-specific**
- exact sklearn CART implementation;
- Gini criterion;
- no depth restriction;
- no pruning;
- no class weighting;
- no SMOTE;
- fixed 0.5 threshold.

Therefore Model 02 is a **literature-grounded baseline, not an exact reproduction** of P02 or P03.


### 3.2.1 Evaluation Protocol

Model 02 uses the identical frozen Part II cohort, split, features, and preprocessing used by Model 01.

This isolates the effect of the learning algorithm.

The primary comparison uses:

- Stratified 5-Fold CV on training data only;
- no resampling;
- fixed threshold 0.5;
- Accuracy, Precision, Recall, Specificity, F1, ROC-AUC, PR-AUC, and Brier score;
- untouched internal test evaluation;
- stratified bootstrap confidence intervals.

Because the tree is intentionally unpruned, we additionally record tree depth, leaf count, node count, and training performance as an overfitting diagnostic.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

MODEL_02_DIR = Path("/kaggle/working/model_02_decision_tree")
MODEL_02_DIR.mkdir(parents=True, exist_ok=True)

decision_tree = DecisionTreeClassifier(
    criterion="gini",
    splitter="best",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    class_weight=None,
    ccp_alpha=0.0,
    random_state=RANDOM_STATE,
)

decision_tree_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", decision_tree),
])

environment_model02 = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "DecisionTreeClassifier",
    "architecture": "CART-style binary classification tree",
    "criterion": "gini",
    "splitter": "best",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": None,
    "class_weight": None,
    "ccp_alpha": 0.0,
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "cv_folds": 5,
}

with open(MODEL_02_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(environment_model02, f, indent=2)

print(json.dumps(environment_model02, indent=2))


### 3.2.2 Five-Fold Cross-Validation

As with Model 01, the complete preprocessing + Decision Tree pipeline is cross-validated.

Each fold independently learns:
- categorical missing handling;
- one-hot encoding;
- numeric imputation;
- scaling;
- tree splits.

No validation-fold or test-set information is used to fit the tree.


In [ ]:
dt_cv_raw = cross_validate(
    decision_tree_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

dt_fold_records = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(dt_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(dt_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(dt_cv_raw["score_time"][fold])
    dt_fold_records.append(row)

dt_cv_folds = pd.DataFrame(dt_fold_records)

dt_summary_records = []
for metric in metric_order:
    values = dt_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    se = sd / sqrt(len(values))
    margin = t_critical_df4_95 * se
    dt_summary_records.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

dt_cv_summary = pd.DataFrame(dt_summary_records)

display(dt_cv_folds)
display(dt_cv_summary)

dt_cv_folds.to_csv(MODEL_02_DIR / "cv_folds.csv", index=False)
dt_cv_summary.to_csv(MODEL_02_DIR / "cv_summary.csv", index=False)


### 3.2.3 Fit the Frozen Tree on the Full Training Set

The Decision Tree is fit once on the complete training partition using the fixed configuration above.

We record tree complexity because unrestricted depth can produce a highly complex model and is a known reason single trees can overfit compared with ensemble methods.


In [ ]:
decision_tree_pipeline.fit(X_train, y_train)

fitted_tree = decision_tree_pipeline.named_steps["model"]

tree_complexity = {
    "max_depth": int(fitted_tree.get_depth()),
    "n_leaves": int(fitted_tree.get_n_leaves()),
    "node_count": int(fitted_tree.tree_.node_count),
}

train_prob_dt = decision_tree_pipeline.predict_proba(X_train)[:, 1]
train_pred_dt = (train_prob_dt >= 0.5).astype(int)

tree_complexity["train_accuracy"] = float(accuracy_score(y_train, train_pred_dt))
tree_complexity["train_roc_auc"] = float(roc_auc_score(y_train, train_prob_dt))

with open(MODEL_02_DIR / "tree_complexity.json", "w", encoding="utf-8") as f:
    json.dump(tree_complexity, f, indent=2)

print(json.dumps(tree_complexity, indent=2))


### 3.2.4 Untouched Internal Test Evaluation

The internal test set is evaluated at the fixed probability threshold of 0.5.

No tree parameter, depth, pruning rule, or threshold is changed after viewing these results.


In [ ]:
y_test_prob_dt = decision_tree_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_dt = (y_test_prob_dt >= 0.5).astype(int)

tn_dt, fp_dt, fn_dt, tp_dt = confusion_matrix(y_test, y_test_pred_dt, labels=[0, 1]).ravel()

dt_test_point_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_dt)),
    "precision": float(precision_score(y_test, y_test_pred_dt, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_dt)),
    "specificity": float(tn_dt / (tn_dt + fp_dt)),
    "f1": float(f1_score(y_test, y_test_pred_dt)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_dt)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_dt)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_dt)),
    "tn": int(tn_dt),
    "fp": int(fp_dt),
    "fn": int(fn_dt),
    "tp": int(tp_dt),
    "threshold": 0.5,
}

print(json.dumps(dt_test_point_metrics, indent=2))

with open(MODEL_02_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(dt_test_point_metrics, f, indent=2)


### 3.2.5 Stratified Bootstrap Confidence Intervals

The same fixed-test bootstrap protocol used for Logistic Regression is reused for Decision Tree so uncertainty estimates are directly comparable.


In [ ]:
BOOTSTRAP_REPS_DT = 200
BOOTSTRAP_SEED_DT = 20251004

rng_dt = np.random.default_rng(BOOTSTRAP_SEED_DT)

dt_bootstrap_rows = []
for b in range(BOOTSTRAP_REPS_DT):
    sample_pos = rng_dt.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng_dt.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng_dt.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_dt[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()

    dt_bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

dt_bootstrap_df = pd.DataFrame(dt_bootstrap_rows)

dt_test_summary_rows = []
for metric in ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc", "brier_score"]:
    values = dt_bootstrap_df[metric].to_numpy()
    dt_test_summary_rows.append({
        "metric": metric,
        "point_estimate": dt_test_point_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

dt_test_summary = pd.DataFrame(dt_test_summary_rows)

display(dt_test_summary)

dt_bootstrap_df.to_csv(MODEL_02_DIR / "bootstrap_metrics.csv", index=False)
dt_test_summary.to_csv(MODEL_02_DIR / "test_summary.csv", index=False)


### 3.2.6 Diagnostic Curves, Confusion Matrix, and Tree Importance

ROC/PR curves and confusion matrix are saved for direct later comparison.

Impurity-based tree feature importance is exported only as a **model diagnostic**. It is not treated as causal evidence and is not used to change the frozen feature set.


In [ ]:
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_test_prob_dt)
precision_curve_dt, recall_curve_dt, _ = precision_recall_curve(y_test, y_test_prob_dt)

fig = plt.figure(figsize=(6, 5))
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC={dt_test_point_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 02 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_02_DIR / "roc_curve.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(6, 5))
plt.plot(recall_curve_dt, precision_curve_dt, label=f"Decision Tree (AP={dt_test_point_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 02 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_02_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm_dt = np.array([[tn_dt, fp_dt], [fn_dt, tp_dt]])
fig = plt.figure(figsize=(5, 4))
plt.imshow(cm_dt)
plt.title("Model 02 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0, 1], ["No diabetes", "Diabetes"])
plt.yticks([0, 1], ["No diabetes", "Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_dt[i, j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_02_DIR / "confusion_matrix.png", dpi=160)
plt.show()

dt_feature_names = decision_tree_pipeline.named_steps["preprocess"].get_feature_names_out()
dt_importances = fitted_tree.feature_importances_

dt_importance_table = (
    pd.DataFrame({
        "transformed_feature": dt_feature_names,
        "gini_importance": dt_importances,
    })
    .sort_values("gini_importance", ascending=False)
)

display(dt_importance_table.head(20))
dt_importance_table.to_csv(MODEL_02_DIR / "feature_importance_transformed.csv", index=False)


### 3.2.7 Model 02 Freeze Point

After this run, Model 02 is frozen as:

> **Unpruned Gini CART-style Decision Tree, sklearn defaults made explicit, no class weighting, no resampling, threshold=0.5.**

Its scientific role is to test whether a single non-linear hierarchical partitioning model improves on the linear Logistic Regression baseline, while also exposing the instability/overfitting risk that motivates later ensemble tree methods.

No post-test pruning or depth tuning is allowed for Model 02. Any tuned/pruned tree would be a separate experiment.


## 3.3 Model 03 — Random Forest

### Why Random Forest follows Decision Tree

Random Forest is the natural ensemble extension of the single Decision Tree baseline.

Direct BRFSS diabetes precedents include:

- **Xie et al. (2019), Preventing Chronic Disease / CDC** — evaluated Random Forest on BRFSS 2014. DOI: `10.5888/pcd16.190109`.
- **Muhammad et al. (2025), Journal of Primary Care & Community Health** — evaluated Random Forest on BRFSS 2023 and explicitly described ensemble averaging as a way to improve stability and reduce overfitting relative to a single tree. DOI: `10.1177/21501319251400546`.

### Architecture used here

A Random Forest trains multiple decision trees on bootstrap samples and averages their class probabilities. At each split, only a random subset of features is considered, decorrelating the trees.

This study uses a fixed sklearn default-style forest:

```text
n_estimators          = 100
criterion             = gini
max_depth             = None
min_samples_split     = 2
min_samples_leaf      = 1
max_features          = sqrt
bootstrap             = True
class_weight          = None
ccp_alpha             = 0.0
random_state          = 42
n_jobs                = -1
threshold             = 0.5
```

### Provenance

**Literature-supported**
- Random Forest as a BRFSS diabetes classifier;
- ensemble-tree role;
- ensemble averaging for improved stability/reduced single-tree overfitting;
- 5-fold CV and held-out evaluation.

**Study-specific**
- exact sklearn configuration;
- 100 trees;
- `sqrt` feature subsampling;
- unpruned component trees;
- no class weighting;
- no resampling;
- threshold 0.5.

Therefore Model 03 is a **literature-grounded Random Forest baseline**, not an exact reproduction of a tuned paper configuration.


### 3.3.1 Evaluation Protocol

To preserve a fair within-study comparison, Random Forest uses exactly the same:

- BRFSS 2025 target cohort;
- frozen 20 predictors;
- 80/20 stratified split;
- Part II preprocessing;
- Stratified 5-Fold CV;
- natural class distribution;
- evaluation metrics;
- untouched test set.

The preprocessing pipeline remains identical even though tree models do not require standardization. This keeps the benchmark input representation fixed across model families.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

MODEL_03_DIR = Path("/kaggle/working/model_03_random_forest")
MODEL_03_DIR.mkdir(parents=True, exist_ok=True)

random_forest = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    ccp_alpha=0.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

random_forest_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", random_forest),
])

environment_model03 = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "RandomForestClassifier",
    "architecture": "bootstrap-aggregated randomized decision-tree ensemble",
    "n_estimators": 100,
    "criterion": "gini",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
    "class_weight": None,
    "ccp_alpha": 0.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "threshold": 0.5,
    "cv_folds": 5,
}

with open(MODEL_03_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(environment_model03, f, indent=2)

print(json.dumps(environment_model03, indent=2))


### 3.3.2 Five-Fold Cross-Validation

The full preprocessing + Random Forest pipeline is evaluated under the same five stratified folds used for the previous models.

Each fold learns preprocessing and all 100 trees only from that fold's fitting partition.


In [ ]:
rf_cv_raw = cross_validate(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

rf_fold_records = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(rf_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(rf_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(rf_cv_raw["score_time"][fold])
    rf_fold_records.append(row)

rf_cv_folds = pd.DataFrame(rf_fold_records)

rf_summary_records = []
for metric in metric_order:
    values = rf_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    se = sd / sqrt(len(values))
    margin = t_critical_df4_95 * se
    rf_summary_records.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

rf_cv_summary = pd.DataFrame(rf_summary_records)

display(rf_cv_folds)
display(rf_cv_summary)

rf_cv_folds.to_csv(MODEL_03_DIR / "cv_folds.csv", index=False)
rf_cv_summary.to_csv(MODEL_03_DIR / "cv_summary.csv", index=False)


### 3.3.3 Fit the Frozen Forest and Inspect Ensemble Complexity

After CV, the same fixed forest is fit once on the complete training partition.

We record aggregate component-tree complexity to compare with the extreme overfitting observed in Model 02. Individual Random Forest trees can still be deep; the key architectural difference is averaging many decorrelated trees rather than relying on one tree.


In [ ]:
random_forest_pipeline.fit(X_train, y_train)

fitted_forest = random_forest_pipeline.named_steps["model"]

tree_depths = np.array([est.get_depth() for est in fitted_forest.estimators_])
tree_leaves = np.array([est.get_n_leaves() for est in fitted_forest.estimators_])
tree_nodes = np.array([est.tree_.node_count for est in fitted_forest.estimators_])

train_prob_rf = random_forest_pipeline.predict_proba(X_train)[:, 1]
train_pred_rf = (train_prob_rf >= 0.5).astype(int)

forest_complexity = {
    "n_estimators": int(len(fitted_forest.estimators_)),
    "mean_tree_depth": float(tree_depths.mean()),
    "min_tree_depth": int(tree_depths.min()),
    "max_tree_depth": int(tree_depths.max()),
    "mean_leaves_per_tree": float(tree_leaves.mean()),
    "mean_nodes_per_tree": float(tree_nodes.mean()),
    "train_accuracy": float(accuracy_score(y_train, train_pred_rf)),
    "train_roc_auc": float(roc_auc_score(y_train, train_prob_rf)),
}

with open(MODEL_03_DIR / "forest_complexity.json", "w", encoding="utf-8") as f:
    json.dump(forest_complexity, f, indent=2)

print(json.dumps(forest_complexity, indent=2))


### 3.3.4 Untouched Internal Test Evaluation

The test set is evaluated once at the fixed 0.5 threshold.

No forest parameter or threshold is changed after test metrics are observed.


In [ ]:
y_test_prob_rf = random_forest_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_rf = (y_test_prob_rf >= 0.5).astype(int)

tn_rf, fp_rf, fn_rf, tp_rf = confusion_matrix(y_test, y_test_pred_rf, labels=[0, 1]).ravel()

rf_test_point_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_rf)),
    "precision": float(precision_score(y_test, y_test_pred_rf, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_rf)),
    "specificity": float(tn_rf / (tn_rf + fp_rf)),
    "f1": float(f1_score(y_test, y_test_pred_rf)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_rf)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_rf)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_rf)),
    "tn": int(tn_rf),
    "fp": int(fp_rf),
    "fn": int(fn_rf),
    "tp": int(tp_rf),
    "threshold": 0.5,
}

print(json.dumps(rf_test_point_metrics, indent=2))

with open(MODEL_03_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(rf_test_point_metrics, f, indent=2)


### 3.3.5 Stratified Bootstrap Confidence Intervals

The same 200-replicate stratified bootstrap protocol is applied to the fixed Random Forest test predictions.


In [ ]:
BOOTSTRAP_REPS_RF = 200
BOOTSTRAP_SEED_RF = 20251005

rng_rf = np.random.default_rng(BOOTSTRAP_SEED_RF)

rf_bootstrap_rows = []
for b in range(BOOTSTRAP_REPS_RF):
    sample_pos = rng_rf.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng_rf.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng_rf.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_rf[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()

    rf_bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

rf_bootstrap_df = pd.DataFrame(rf_bootstrap_rows)

rf_test_summary_rows = []
for metric in ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc", "brier_score"]:
    values = rf_bootstrap_df[metric].to_numpy()
    rf_test_summary_rows.append({
        "metric": metric,
        "point_estimate": rf_test_point_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

rf_test_summary = pd.DataFrame(rf_test_summary_rows)

display(rf_test_summary)

rf_bootstrap_df.to_csv(MODEL_03_DIR / "bootstrap_metrics.csv", index=False)
rf_test_summary.to_csv(MODEL_03_DIR / "test_summary.csv", index=False)


### 3.3.6 Diagnostic Curves, Confusion Matrix, and Forest Importance

ROC/PR curves and the confusion matrix are saved for later model-comparison figures.

Impurity-based Random Forest feature importance is exported as a diagnostic only. It is not causal evidence and is not used to modify the frozen predictor set.


In [ ]:
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_test_prob_rf)
precision_curve_rf, recall_curve_rf, _ = precision_recall_curve(y_test, y_test_prob_rf)

fig = plt.figure(figsize=(6, 5))
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={rf_test_point_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 03 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_03_DIR / "roc_curve.png", dpi=160)
plt.show()

fig = plt.figure(figsize=(6, 5))
plt.plot(recall_curve_rf, precision_curve_rf, label=f"Random Forest (AP={rf_test_point_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 03 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_03_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm_rf = np.array([[tn_rf, fp_rf], [fn_rf, tp_rf]])
fig = plt.figure(figsize=(5, 4))
plt.imshow(cm_rf)
plt.title("Model 03 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0, 1], ["No diabetes", "Diabetes"])
plt.yticks([0, 1], ["No diabetes", "Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_rf[i, j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_03_DIR / "confusion_matrix.png", dpi=160)
plt.show()

rf_feature_names = random_forest_pipeline.named_steps["preprocess"].get_feature_names_out()
rf_importances = fitted_forest.feature_importances_

rf_importance_table = (
    pd.DataFrame({
        "transformed_feature": rf_feature_names,
        "gini_importance": rf_importances,
    })
    .sort_values("gini_importance", ascending=False)
)

display(rf_importance_table.head(20))
rf_importance_table.to_csv(MODEL_03_DIR / "feature_importance_transformed.csv", index=False)


### 3.3.7 Model 03 Freeze Point

After this verified run, Model 03 is frozen as:

> **100-tree bootstrap Random Forest, Gini splits, sqrt feature sampling, unrestricted component-tree depth, no class weighting, no resampling, threshold=0.5.**

Its role is to test whether ensemble averaging substantially improves generalization over the overfit single Decision Tree baseline.

No post-test change to tree count, depth, feature sampling, class weights, or threshold is permitted. Any tuned Random Forest must be recorded as a separate experiment.


## 3.4–3.7 Remaining Frozen Model Families

Models 04–07 continue **inside this same notebook**. They reuse the Part II cohort, train/test partition, preprocessing transformer definition, CV folds, scoring rules, and held-out test set already established above.

The model-specific sections below therefore define only the estimator architecture and its evaluation. They do **not** reload BRFSS, rebuild the cohort, or create a different split.


In [ ]:
# Shared imports/helpers for Models 04–07 and the paired comparison.
from itertools import combinations
from scipy.stats import binomtest
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
import xgboost as xgb

def to_dense(X_matrix):
    return X_matrix.toarray() if hasattr(X_matrix, "toarray") else np.asarray(X_matrix)

dense_transformer = FunctionTransformer(
    to_dense,
    accept_sparse=True,
    feature_names_out="one-to-one",
)

print("Additional frozen model families ready.")


## 3.4 Model 04 — Gradient Boosting

## BRFSS 2025 Diabetes Classification

**Purpose:** independently execute and freeze the fourth classifier in the benchmark without rerunning Models 01–03.

### Literature provenance

**BRFSS application:** Muhammad et al. (2025), *Journal of Primary Care & Community Health*, DOI `10.1177/21501319251400546`, evaluated Gradient Boosting as one of seven classifiers on BRFSS 2023 using stratified 5-fold CV.

**Architecture source:** Friedman (2001), *Greedy Function Approximation: A Gradient Boosting Machine*, *The Annals of Statistics*, DOI `10.1214/aos/1013203451`.

Gradient boosting builds an additive classifier sequentially. Each new shallow regression tree is fit to the negative gradient of the current loss, so later stages correct errors left by earlier stages.

This notebook uses the frozen Part II cohort, features, preprocessing, split, and evaluation protocol. It does **not** retune prior models.


### 3.4.1 Frozen Study Protocol

The following are inherited unchanged from Part II:

- target: `DIABETE4=1` versus `DIABETE4=3`;
- 20 frozen predictors;
- 80/20 stratified split with `random_state=42`;
- categorical missing values represented explicitly and one-hot encoded;
- continuous missing values median-imputed;
- continuous predictors standardized;
- Stratified 5-Fold CV on training data only;
- no SMOTE or class resampling in the primary benchmark;
- untouched test set evaluated once after model specification is frozen.

The test set is never used to select Gradient Boosting hyperparameters.


### 3.4.2 Model Architecture

The fixed baseline is:

```python
GradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.1,
    n_estimators=100,
    subsample=1.0,
    criterion="friedman_mse",
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    max_features=None,
    random_state=42,
)
```

### Provenance classification

**Literature/architecture-supported**
- Gradient Boosting model family;
- sequential additive trees minimizing classification loss;
- use on BRFSS diabetes;
- 5-fold CV and held-out evaluation.

**Study-specific fixed choices**
- 100 stages;
- learning rate 0.1;
- depth-3 weak learners;
- no stochastic subsampling;
- no class weighting/resampling;
- threshold 0.5.

These values are fixed before test evaluation and are not claimed to reproduce the tuned P03 configuration.


In [ ]:
MODEL_04_DIR = Path("/kaggle/working/model_04_gradient_boosting")
MODEL_04_DIR.mkdir(parents=True, exist_ok=True)

gradient_boosting = GradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.1,
    n_estimators=100,
    subsample=1.0,
    criterion="friedman_mse",
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=3,
    max_features=None,
    random_state=RANDOM_STATE,
)

gb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", gradient_boosting),
])

gb_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "GradientBoostingClassifier",
    "architecture": "forward stage-wise additive regression-tree ensemble",
    "loss": "log_loss",
    "learning_rate": 0.1,
    "n_estimators": 100,
    "subsample": 1.0,
    "criterion": "friedman_mse",
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_depth": 3,
    "max_features": None,
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_04_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(gb_config, f, indent=2)

print(json.dumps(gb_config, indent=2))


### 3.4.3 Five-Fold Cross-Validation

The full preprocessing + Gradient Boosting pipeline is fitted independently inside each training fold.

No preprocessing statistic or tree is learned from the validation fold or final test set.


In [ ]:
gb_cv_raw = cross_validate(
    gb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(gb_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(gb_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(gb_cv_raw["score_time"][fold])
    fold_rows.append(row)

gb_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = gb_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

gb_cv_summary = pd.DataFrame(summary_rows)
display(gb_cv_folds)
display(gb_cv_summary)

gb_cv_folds.to_csv(MODEL_04_DIR / "cv_folds.csv", index=False)
gb_cv_summary.to_csv(MODEL_04_DIR / "cv_summary.csv", index=False)


### 3.4.4 Fit Frozen Model and Evaluate the Untouched Test Set

After CV, the exact same fixed configuration is fitted once on all training respondents.

The test set is evaluated once at probability threshold 0.5. No hyperparameter or threshold is changed afterward.


In [ ]:
gb_pipeline.fit(X_train, y_train)

y_test_prob_gb = gb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_gb = (y_test_prob_gb >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_gb, labels=[0, 1]).ravel()

gb_test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_gb)),
    "precision": float(precision_score(y_test, y_test_pred_gb, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_gb)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_gb)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_gb)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_gb)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_gb)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "threshold": 0.5,
}
print(json.dumps(gb_test_metrics, indent=2))

with open(MODEL_04_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(gb_test_metrics, f, indent=2)


### 3.4.5 Stratified Bootstrap Confidence Intervals

As with Models 01–03, uncertainty on fixed test predictions is estimated with 200 stratified bootstrap replicates. This procedure estimates uncertainty only; it does not tune the model.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251006

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_gb[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)
    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()

    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

gb_bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc","brier_score"]:
    values = gb_bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": gb_test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

gb_test_summary = pd.DataFrame(test_summary_rows)
display(gb_test_summary)

gb_bootstrap_df.to_csv(MODEL_04_DIR / "bootstrap_metrics.csv", index=False)
gb_test_summary.to_csv(MODEL_04_DIR / "gb_test_summary.csv", index=False)


### 3.4.6 Diagnostics

ROC/PR curves and the fixed-threshold confusion matrix are saved.

Gradient Boosting impurity importance is exported as a diagnostic only. It is not causal evidence and is not used to reselect predictors.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_gb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_gb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Gradient Boosting (AUC={gb_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 04 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Gradient Boosting (AP={gb_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 04 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 04 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_04_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = gb_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = gb_pipeline.named_steps["model"].feature_importances_
importance_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "gini_importance": importances,
}).sort_values("gini_importance", ascending=False)

display(importance_table.head(20))
importance_table.to_csv(MODEL_04_DIR / "feature_importance_transformed.csv", index=False)


### 3.4.7 Freeze Rule

After this notebook completes successfully, Model 04 is frozen as:

> **100-stage Gradient Boosting classifier, log-loss objective, learning rate 0.1, depth-3 regression-tree weak learners, no resampling, threshold 0.5.**

No post-test change to stage count, learning rate, tree depth, threshold, preprocessing, or feature set is allowed. Any tuned Gradient Boosting configuration must be versioned as a separate experiment.


## 3.5 Model 05 — XGBoost

## BRFSS 2025 Diabetes Classification

**Purpose:** execute and freeze the fifth classifier without rerunning Models 01–04.

### Literature provenance

**BRFSS application 1:** Muhammad et al. (2025), *Journal of Primary Care & Community Health*, DOI `10.1177/21501319251400546`, included XGBoost among seven classifiers for self-reported diabetes on BRFSS 2023.

**BRFSS application 2:** Li, Peng, & Peng (2024), *PLOS ONE*, DOI `10.1371/journal.pone.0311222`, used XGBoost on BRFSS 2021 and later optimized it with a genetic algorithm.

**Architecture source:** Chen & Guestrin (2016), *XGBoost: A Scalable Tree Boosting System*, KDD '16, DOI `10.1145/2939672.2939785`.

This notebook evaluates a **plain fixed XGBoost baseline**, not GA-XGBoost and not a tuned reproduction of either BRFSS paper.


### 3.5.1 Frozen Study Protocol

Inherited unchanged from Part II:

- target: `DIABETE4=1` vs `DIABETE4=3`;
- 20 frozen predictors;
- 80/20 stratified split, `random_state=42`;
- categorical missing category + one-hot encoding;
- continuous median imputation + scaling;
- Stratified 5-Fold CV on training data only;
- natural class distribution;
- no SMOTE/undersampling;
- untouched test set used only after specification is frozen.

This allows a direct within-study comparison with Models 01–04.


### 3.5.2 Model Architecture

XGBoost extends gradient tree boosting with a regularized objective and systems optimizations for scalable tree construction. The original Chen & Guestrin formulation includes regularization on tree complexity and second-order optimization of the boosting objective.

The fixed baseline used here is:

```python
XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    booster="gbtree",
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    min_child_weight=1,
    gamma=0.0,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
```

### Provenance classification

**Literature-supported**
- XGBoost model family and regularized boosting architecture;
- direct BRFSS diabetes use in P03 and P05;
- stratified CV / held-out evaluation.

**Study-specific fixed choices**
- 100 trees;
- learning rate 0.3;
- max depth 6;
- default-style L2 regularization;
- histogram tree builder for efficient execution;
- no SMOTE;
- threshold 0.5.

This is a plain XGBoost baseline, not GA-XGBoost and not a tuned reproduction.


In [ ]:
MODEL_05_DIR = Path("/kaggle/working/model_05_xgboost")
MODEL_05_DIR.mkdir(parents=True, exist_ok=True)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    booster="gbtree",
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    min_child_weight=1,
    gamma=0.0,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)

xgb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", xgb_model),
])

xgb_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "model": "XGBClassifier",
    "architecture": "regularized gradient-boosted decision-tree ensemble",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "booster": "gbtree",
    "n_estimators": 100,
    "learning_rate": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "gamma": 0.0,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_05_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(xgb_config, f, indent=2)

print(json.dumps(xgb_config, indent=2))


### 3.5.3 Five-Fold Cross-Validation

The complete preprocessing + XGBoost pipeline is re-fit independently inside each training fold.

No validation-fold or test-set information is used to select trees or preprocessing statistics.


In [ ]:
xgb_cv_raw = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(xgb_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(xgb_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(xgb_cv_raw["score_time"][fold])
    fold_rows.append(row)

xgb_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = xgb_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

xgb_cv_summary = pd.DataFrame(summary_rows)
display(xgb_cv_folds)
display(xgb_cv_summary)

xgb_cv_folds.to_csv(MODEL_05_DIR / "cv_folds.csv", index=False)
xgb_cv_summary.to_csv(MODEL_05_DIR / "cv_summary.csv", index=False)


### 3.5.4 Fit Frozen Model and Evaluate the Untouched Test Set

After CV, the exact same fixed configuration is fit once on the full training set.

The test set is then evaluated once at threshold 0.5. No post-test parameter adjustment is permitted.


In [ ]:
xgb_pipeline.fit(X_train, y_train)

y_test_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_xgb = (y_test_prob_xgb >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_xgb, labels=[0, 1]).ravel()

xgb_test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_xgb)),
    "precision": float(precision_score(y_test, y_test_pred_xgb, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_xgb)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_xgb)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_xgb)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_xgb)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_xgb)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "threshold": 0.5,
}
print(json.dumps(xgb_test_metrics, indent=2))

with open(MODEL_05_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(xgb_test_metrics, f, indent=2)


### 3.5.5 Stratified Bootstrap Confidence Intervals

Two hundred class-stratified bootstrap samples estimate uncertainty around the already-fixed held-out predictions. They are not used for tuning.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251007

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_xgb[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()
    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

xgb_bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc","brier_score"]:
    values = xgb_bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": xgb_test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

xgb_test_summary = pd.DataFrame(test_summary_rows)
display(xgb_test_summary)

xgb_bootstrap_df.to_csv(MODEL_05_DIR / "bootstrap_metrics.csv", index=False)
xgb_test_summary.to_csv(MODEL_05_DIR / "xgb_test_summary.csv", index=False)


### 3.5.6 Diagnostics

The ROC curve, precision-recall curve, confusion matrix, and transformed-feature importance are exported.

XGBoost feature importance is a model diagnostic only and is not causal evidence.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_xgb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_xgb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"XGBoost (AUC={xgb_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 05 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"XGBoost (AP={xgb_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 05 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 05 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_05_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = xgb_pipeline.named_steps["preprocess"].get_feature_names_out()
importances = xgb_pipeline.named_steps["model"].feature_importances_
importance_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "xgboost_importance": importances,
}).sort_values("xgboost_importance", ascending=False)

display(importance_table.head(20))
importance_table.to_csv(MODEL_05_DIR / "feature_importance_transformed.csv", index=False)


### 3.5.7 Freeze Rule

After verified execution, Model 05 is frozen as:

> **100-tree XGBoost gbtree classifier, binary logistic objective, learning rate 0.3, max depth 6, L2 regularization 1.0, histogram tree builder, no resampling, threshold 0.5.**

No post-test change to depth, learning rate, regularization, sampling, threshold, preprocessing, or feature set is allowed.

Any GA-XGBoost, tuned XGBoost, class-weighted XGBoost, or resampled XGBoost must be recorded as a separate experiment.


## 3.6 Model 06 — Linear Support Vector Machine

## BRFSS 2025 Diabetes Classification

**Purpose:** execute and freeze the sixth classifier without rerunning Models 01–05.

### Literature provenance

**BRFSS application — Xie et al. (2019), Preventing Chronic Disease / CDC**, DOI `10.5888/pcd16.190109`.

That study evaluated three SVM kernels on BRFSS 2014: linear, RBF, and polynomial. Among their SVM variants, the linear SVM had the highest reported AUC.

**BRFSS application — Muhammad et al. (2025)**, DOI `10.1177/21501319251400546`.

That study included Support Vector Machine among seven BRFSS 2023 diabetes classifiers using stratified 5-fold CV.

**Architecture source — Cortes & Vapnik (1995), Support-Vector Networks**, DOI `10.1007/BF00994018`.

Model 06 intentionally uses a **linear SVM** rather than an RBF kernel because the transformed BRFSS dataset is large and sparse, and linear SVM has direct BRFSS precedent.


### 3.6.1 Frozen Study Protocol

Inherited unchanged from Part II:

- `DIABETE4=1` vs `DIABETE4=3`;
- frozen 20 predictors;
- 80/20 stratified split with `random_state=42`;
- categorical missing category + one-hot encoding;
- continuous median imputation + standardization;
- Stratified 5-Fold CV;
- no SMOTE or class reweighting;
- untouched internal test set.

### Important metric difference

`LinearSVC` does not produce calibrated probabilities.

Therefore:
- Accuracy, Precision, Recall, Specificity, and F1 use predicted labels.
- ROC-AUC and PR-AUC use the continuous `decision_function` margin.
- The class boundary is **decision score = 0**.
- **Brier score is not reported** because applying it to uncalibrated margins would be invalid.

Probability calibration, if desired, must be a separate experiment.


### 3.6.2 Model Architecture

Support Vector Machines seek a separating hyperplane with a large margin between classes, while soft-margin regularization allows classification violations for non-separable data.

The fixed large-sample linear baseline is:

```python
LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual="auto",
    tol=1e-4,
    C=1.0,
    fit_intercept=True,
    class_weight=None,
    random_state=42,
    max_iter=10000,
)
```

### Provenance classification

**Literature-supported**
- SVM family and maximum-margin architecture;
- direct linear SVM use on BRFSS diabetes in P02;
- SVM comparator role in P03.

**Study-specific fixed choices**
- `LinearSVC` implementation;
- L2 + squared-hinge objective;
- `C=1.0`;
- automatic primal/dual selection;
- no class weighting;
- no SMOTE;
- zero-margin class threshold.

No RBF/poly kernel or calibration is added to this primary baseline.


In [ ]:
MODEL_06_DIR = Path("/kaggle/working/model_06_linear_svm")
MODEL_06_DIR.mkdir(parents=True, exist_ok=True)

linear_svm = LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual="auto",
    tol=1e-4,
    C=1.0,
    fit_intercept=True,
    class_weight=None,
    random_state=RANDOM_STATE,
    max_iter=10000,
)

svm_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", linear_svm),
])

svm_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "LinearSVC",
    "architecture": "soft-margin linear support vector classifier",
    "penalty": "l2",
    "loss": "squared_hinge",
    "dual": "auto",
    "tol": 1e-4,
    "C": 1.0,
    "fit_intercept": True,
    "class_weight": None,
    "random_state": RANDOM_STATE,
    "max_iter": 10000,
    "decision_threshold": 0.0,
    "probability_calibration": None,
    "resampling": "none",
    "cv_folds": 5,
}
with open(MODEL_06_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(svm_config, f, indent=2)

print(json.dumps(svm_config, indent=2))


### 3.6.3 Five-Fold Cross-Validation

The full preprocessing + Linear SVM pipeline is fitted independently inside every training fold.

ROC-AUC and PR-AUC are computed from the signed decision margin rather than probabilities.


In [ ]:
svm_cv_raw = cross_validate(
    svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(svm_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(svm_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(svm_cv_raw["score_time"][fold])
    fold_rows.append(row)

svm_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = svm_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

svm_cv_summary = pd.DataFrame(summary_rows)
display(svm_cv_folds)
display(svm_cv_summary)

svm_cv_folds.to_csv(MODEL_06_DIR / "cv_folds.csv", index=False)
svm_cv_summary.to_csv(MODEL_06_DIR / "cv_summary.csv", index=False)


### 3.6.4 Fit Frozen Model and Evaluate the Untouched Test Set

The fixed model is fit once on the full training partition.

Class labels use the natural SVM boundary at decision margin zero. Ranking metrics use the continuous decision margin.


In [ ]:
svm_pipeline.fit(X_train, y_train)

y_test_score_svm = svm_pipeline.decision_function(X_test)
y_test_pred_svm = svm_pipeline.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_svm, labels=[0, 1]).ravel()

svm_test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_svm)),
    "precision": float(precision_score(y_test, y_test_pred_svm, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_svm)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_svm)),
    "roc_auc": float(roc_auc_score(y_test, y_test_score_svm)),
    "pr_auc": float(average_precision_score(y_test, y_test_score_svm)),
    "brier_score": None,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "decision_threshold": 0.0,
}

model = svm_pipeline.named_steps["model"]
svm_test_metrics["n_iter"] = int(np.max(np.atleast_1d(model.n_iter_)))

print(json.dumps(svm_test_metrics, indent=2))

with open(MODEL_06_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(svm_test_metrics, f, indent=2)


### 3.6.5 Stratified Bootstrap Confidence Intervals

The same 200-replicate class-stratified bootstrap is used for all valid Linear SVM metrics.

Brier score is omitted because the SVM margins are not calibrated probabilities.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251008

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    score_b = y_test_score_svm[sample_idx]
    pred_b = (score_b > 0.0).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()
    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, score_b),
        "pr_auc": average_precision_score(y_b, score_b),
    })

svm_bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc"]:
    values = svm_bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": svm_test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

svm_test_summary = pd.DataFrame(test_summary_rows)
display(svm_test_summary)

svm_bootstrap_df.to_csv(MODEL_06_DIR / "bootstrap_metrics.csv", index=False)
svm_test_summary.to_csv(MODEL_06_DIR / "svm_test_summary.csv", index=False)


### 3.6.6 Diagnostics

ROC and precision-recall curves use the SVM decision score.

The transformed-feature coefficients describe the fitted hyperplane. Their magnitudes are model diagnostics, not causal effects.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_score_svm)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_score_svm)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Linear SVM (AUC={svm_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 06 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Linear SVM (AP={svm_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 06 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 06 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_06_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = svm_pipeline.named_steps["preprocess"].get_feature_names_out()
coef = svm_pipeline.named_steps["model"].coef_.ravel()
coef_table = pd.DataFrame({
    "transformed_feature": feature_names,
    "coefficient": coef,
    "abs_coefficient": np.abs(coef),
}).sort_values("abs_coefficient", ascending=False)

display(coef_table.head(20))
coef_table.to_csv(MODEL_06_DIR / "linear_coefficients.csv", index=False)


### 3.6.7 Freeze Rule

After verified execution, Model 06 is frozen as:

> **L2 Linear SVM with squared-hinge loss, C=1.0, automatic primal/dual selection, no class weighting, no resampling, zero-margin decision boundary.**

No post-test change to `C`, loss, class weighting, calibration, decision threshold, preprocessing, or feature set is allowed.

RBF/poly SVMs or probability-calibrated SVMs must be separate experiments.


## 3.7 Model 07 — Gaussian Naive Bayes

## BRFSS 2025 Diabetes Classification

**Purpose:** execute and freeze the seventh and final primary classifier without rerunning Models 01–06.

### Literature provenance

**BRFSS application:** Xie et al. (2019), *Preventing Chronic Disease / CDC*, DOI `10.5888/pcd16.190109`, directly evaluated Gaussian Naive Bayes on BRFSS 2014.

**Architecture reference:** Friedman, Geiger, & Goldszmidt (1997), *Bayesian Network Classifiers*, *Machine Learning*, DOI `10.1023/A:1007465528199`.

Gaussian Naive Bayes provides a computationally efficient probabilistic baseline with a very different inductive bias from the linear-margin and tree-ensemble models already evaluated.


### 3.7.1 Frozen Study Protocol

Inherited unchanged from Part II:

- target: `DIABETE4=1` vs `DIABETE4=3`;
- same 20 predictors;
- 80/20 stratified split with `random_state=42`;
- same missing-value handling;
- same one-hot categorical representation;
- same numeric scaling;
- Stratified 5-Fold CV on training data;
- no SMOTE or class balancing;
- untouched test set.

### Dense conversion

The frozen preprocessing pipeline outputs a sparse matrix because of one-hot encoding.

`GaussianNB` requires dense input, so Model 07 adds only:

```text
frozen Part II preprocessing
        ↓
sparse matrix → dense array
        ↓
GaussianNB
```

No values, categories, scaling rules, or predictors are changed.


### 3.7.2 Model Architecture

Naive Bayes applies Bayes' rule under the simplifying assumption that predictors are conditionally independent given the outcome class.

Gaussian Naive Bayes additionally models each feature's class-conditional distribution as Gaussian.

The fixed baseline is:

```python
GaussianNB(
    priors=None,
    var_smoothing=1e-9,
)
```

### Provenance classification

**Literature-supported**
- Gaussian Naive Bayes directly used on BRFSS diabetes in P02;
- probabilistic Naive Bayes classifier family;
- held-out evaluation.

**Study-specific**
- sklearn `GaussianNB`;
- default empirical class priors;
- `var_smoothing=1e-9`;
- sparse-to-dense adapter;
- no SMOTE;
- probability threshold 0.5.

### Assumption caveat

After one-hot encoding, many transformed features are binary rather than truly Gaussian. Model 07 is therefore treated as a simple probabilistic baseline, not as a claim that the Gaussian feature-distribution assumption is scientifically correct for every BRFSS indicator.


In [ ]:
MODEL_07_DIR = Path("/kaggle/working/model_07_gaussian_naive_bayes")
MODEL_07_DIR.mkdir(parents=True, exist_ok=True)

gnb = GaussianNB(
    priors=None,
    var_smoothing=1e-9,
)

gnb_pipeline = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("to_dense", dense_transformer),
    ("model", gnb),
])

gnb_config = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "model": "GaussianNB",
    "architecture": "Gaussian class-conditional naive Bayes classifier",
    "priors": None,
    "var_smoothing": 1e-9,
    "dense_adapter": True,
    "threshold": 0.5,
    "resampling": "none",
    "cv_folds": 5,
}

with open(MODEL_07_DIR / "environment_and_config.json", "w", encoding="utf-8") as f:
    json.dump(gnb_config, f, indent=2)

print(json.dumps(gnb_config, indent=2))


### 3.7.3 Five-Fold Cross-Validation

The complete preprocessing + dense conversion + GaussianNB pipeline is re-fit independently inside each training fold.

Dense conversion is representation-only: it does not alter feature values.


In [ ]:
gnb_cv_raw = cross_validate(
    gnb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
)

metric_order = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc"]
t_critical_df4_95 = 2.7764451051977987

fold_rows = []
for fold in range(5):
    row = {"fold": fold + 1}
    for metric in metric_order:
        row[metric] = float(gnb_cv_raw[f"test_{metric}"][fold])
    row["fit_time_seconds"] = float(gnb_cv_raw["fit_time"][fold])
    row["score_time_seconds"] = float(gnb_cv_raw["score_time"][fold])
    fold_rows.append(row)

gnb_cv_folds = pd.DataFrame(fold_rows)

summary_rows = []
for metric in metric_order:
    values = gnb_cv_folds[metric].to_numpy()
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    margin = t_critical_df4_95 * (sd / sqrt(len(values)))
    summary_rows.append({
        "metric": metric,
        "mean": mean,
        "std": sd,
        "ci95_lower": max(0.0, mean - margin),
        "ci95_upper": min(1.0, mean + margin),
    })

gnb_cv_summary = pd.DataFrame(summary_rows)
display(gnb_cv_folds)
display(gnb_cv_summary)

gnb_cv_folds.to_csv(MODEL_07_DIR / "cv_folds.csv", index=False)
gnb_cv_summary.to_csv(MODEL_07_DIR / "cv_summary.csv", index=False)


### 3.7.4 Fit Frozen Model and Evaluate the Untouched Test Set

The exact same fixed model is fit once on all training respondents.

The test set is evaluated once at probability threshold 0.5. No prior, smoothing, or threshold value is changed afterward.


In [ ]:
gnb_pipeline.fit(X_train, y_train)

y_test_prob_gnb = gnb_pipeline.predict_proba(X_test)[:, 1]
y_test_pred_gnb = (y_test_prob_gnb >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred_gnb, labels=[0, 1]).ravel()

gnb_test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_test_pred_gnb)),
    "precision": float(precision_score(y_test, y_test_pred_gnb, zero_division=0)),
    "recall": float(recall_score(y_test, y_test_pred_gnb)),
    "specificity": float(tn / (tn + fp)),
    "f1": float(f1_score(y_test, y_test_pred_gnb)),
    "roc_auc": float(roc_auc_score(y_test, y_test_prob_gnb)),
    "pr_auc": float(average_precision_score(y_test, y_test_prob_gnb)),
    "brier_score": float(brier_score_loss(y_test, y_test_prob_gnb)),
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "threshold": 0.5,
}

print(json.dumps(gnb_test_metrics, indent=2))

with open(MODEL_07_DIR / "test_point_metrics.json", "w", encoding="utf-8") as f:
    json.dump(gnb_test_metrics, f, indent=2)


### 3.7.5 Stratified Bootstrap Confidence Intervals

Two hundred class-stratified bootstrap samples quantify uncertainty around the fixed held-out predictions.


In [ ]:
BOOTSTRAP_REPS = 200
BOOTSTRAP_SEED = 20251009

rng = np.random.default_rng(BOOTSTRAP_SEED)
y_test_np = y_test.to_numpy()
pos_idx = np.flatnonzero(y_test_np == 1)
neg_idx = np.flatnonzero(y_test_np == 0)

bootstrap_rows = []
for b in range(BOOTSTRAP_REPS):
    sample_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
    sample_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
    sample_idx = np.concatenate([sample_pos, sample_neg])
    rng.shuffle(sample_idx)

    y_b = y_test_np[sample_idx]
    p_b = y_test_prob_gnb[sample_idx]
    pred_b = (p_b >= 0.5).astype(int)

    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_b, pred_b, labels=[0, 1]).ravel()
    bootstrap_rows.append({
        "replicate": b + 1,
        "accuracy": accuracy_score(y_b, pred_b),
        "precision": precision_score(y_b, pred_b, zero_division=0),
        "recall": recall_score(y_b, pred_b),
        "specificity": tn_b / (tn_b + fp_b),
        "f1": f1_score(y_b, pred_b),
        "roc_auc": roc_auc_score(y_b, p_b),
        "pr_auc": average_precision_score(y_b, p_b),
        "brier_score": brier_score_loss(y_b, p_b),
    })

gnb_bootstrap_df = pd.DataFrame(bootstrap_rows)
test_summary_rows = []
for metric in ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc","brier_score"]:
    values = gnb_bootstrap_df[metric].to_numpy()
    test_summary_rows.append({
        "metric": metric,
        "point_estimate": gnb_test_metrics[metric],
        "bootstrap_mean": float(values.mean()),
        "ci95_lower": float(np.quantile(values, 0.025)),
        "ci95_upper": float(np.quantile(values, 0.975)),
    })

gnb_test_summary = pd.DataFrame(test_summary_rows)
display(gnb_test_summary)

gnb_bootstrap_df.to_csv(MODEL_07_DIR / "bootstrap_metrics.csv", index=False)
gnb_test_summary.to_csv(MODEL_07_DIR / "gnb_test_summary.csv", index=False)


### 3.7.6 Diagnostics

ROC, precision-recall, confusion matrix, and class-conditional Gaussian parameters are saved.

The learned means/variances are descriptive model parameters only and are not causal effects.


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob_gnb)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_prob_gnb)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Gaussian NB (AUC={gnb_test_metrics['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 07 — ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "roc_curve.png", dpi=160)
plt.show()

plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve, label=f"Gaussian NB (AP={gnb_test_metrics['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label=f"Class prevalence={y_test.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Model 07 — Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "precision_recall_curve.png", dpi=160)
plt.show()

cm = np.array([[tn, fp],[fn, tp]])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Model 07 — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks([0,1], ["No diabetes","Diabetes"])
plt.yticks([0,1], ["No diabetes","Diabetes"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(MODEL_07_DIR / "confusion_matrix.png", dpi=160)
plt.show()

feature_names = gnb_pipeline.named_steps["preprocess"].get_feature_names_out()
model = gnb_pipeline.named_steps["model"]
params = pd.DataFrame({
    "transformed_feature": feature_names,
    "mean_no_diabetes": model.theta_[0],
    "mean_diabetes": model.theta_[1],
    "variance_no_diabetes": model.var_[0],
    "variance_diabetes": model.var_[1],
})
params.to_csv(MODEL_07_DIR / "gaussian_parameters.csv", index=False)
display(params.head(20))


### 3.7.7 Freeze Rule

After verified execution, Model 07 is frozen as:

> **Gaussian Naive Bayes with empirical class priors, var_smoothing=1e-9, dense adapter after frozen preprocessing, no resampling, threshold=0.5.**

No post-test changes are allowed to priors, smoothing, threshold, preprocessing, or feature set.

This completes the **seven-model primary benchmark**. Any KNN, neural network, calibrated model, tuned model, or resampling variant belongs to sensitivity/secondary experiments rather than the primary comparison.


# Part IV — Consolidated Seven-Model Comparison and Paired Statistical Analysis

The seven primary models are now frozen. This section **does not fit them again**.

Instead, it reuses the respondent-level held-out predictions/scores already generated in Sections 3.1–3.7. This has three advantages:

- every comparison uses the exact same 68,508 held-out respondents;
- no second implementation of a model can silently drift from its frozen configuration;
- the comparison directly benefits from work already performed in the model sections.

Before any paired inference, the notebook reconciles the current point metrics with the previously verified frozen benchmark. Any material drift stops execution.


In [ ]:
COMPARISON_DIR = Path("/kaggle/working/model_comparison")
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = [
    "logistic_regression",
    "decision_tree",
    "random_forest",
    "gradient_boosting",
    "xgboost",
    "linear_svm",
    "gaussian_naive_bayes",
]
DISPLAY_NAMES = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "gradient_boosting": "Gradient Boosting",
    "xgboost": "XGBoost",
    "linear_svm": "Linear SVM",
    "gaussian_naive_bayes": "Gaussian Naive Bayes",
}
MODEL_FAMILY = {
    "logistic_regression": "linear probabilistic",
    "decision_tree": "single tree",
    "random_forest": "bagged tree ensemble",
    "gradient_boosting": "sequential boosting ensemble",
    "xgboost": "regularized boosting ensemble",
    "linear_svm": "maximum-margin linear",
    "gaussian_naive_bayes": "probabilistic conditional-independence",
}
PROVENANCE = {
    "logistic_regression": "P01;P02;P03",
    "decision_tree": "P02;P03",
    "random_forest": "P02;P03",
    "gradient_boosting": "P03;Friedman2001",
    "xgboost": "P03;P05;ChenGuestrin2016",
    "linear_svm": "P02;P03;CortesVapnik1995",
    "gaussian_naive_bayes": "P02;FriedmanGeigerGoldszmidt1997",
}
THRESHOLD = {
    "logistic_regression": 0.5,
    "decision_tree": 0.5,
    "random_forest": 0.5,
    "gradient_boosting": 0.5,
    "xgboost": 0.5,
    "linear_svm": 0.0,
    "gaussian_naive_bayes": 0.5,
}

scores = {
    "logistic_regression": np.asarray(y_test_prob),
    "decision_tree": np.asarray(y_test_prob_dt),
    "random_forest": np.asarray(y_test_prob_rf),
    "gradient_boosting": np.asarray(y_test_prob_gb),
    "xgboost": np.asarray(y_test_prob_xgb),
    "linear_svm": np.asarray(y_test_score_svm),
    "gaussian_naive_bayes": np.asarray(y_test_prob_gnb),
}
preds = {
    "logistic_regression": np.asarray(y_test_pred),
    "decision_tree": np.asarray(y_test_pred_dt),
    "random_forest": np.asarray(y_test_pred_rf),
    "gradient_boosting": np.asarray(y_test_pred_gb),
    "xgboost": np.asarray(y_test_pred_xgb),
    "linear_svm": np.asarray(y_test_pred_svm),
    "gaussian_naive_bayes": np.asarray(y_test_pred_gnb),
}
cv_tables = {
    "logistic_regression": logreg_cv_summary,
    "decision_tree": dt_cv_summary,
    "random_forest": rf_cv_summary,
    "gradient_boosting": gb_cv_summary,
    "xgboost": xgb_cv_summary,
    "linear_svm": svm_cv_summary,
    "gaussian_naive_bayes": gnb_cv_summary,
}
test_ci_tables = {
    "logistic_regression": logreg_test_summary,
    "decision_tree": dt_test_summary,
    "random_forest": rf_test_summary,
    "gradient_boosting": gb_test_summary,
    "xgboost": xgb_test_summary,
    "linear_svm": svm_test_summary,
    "gaussian_naive_bayes": gnb_test_summary,
}

# Previously verified frozen point estimates. These are a guardrail, not a new tuning target.
FROZEN = {
    "logistic_regression": {"accuracy":0.8558270566941087,"f1":0.2831845562087234,"roc_auc":0.8261678122652486,"pr_auc":0.44663290841217973},
    "decision_tree": {"accuracy":0.78615636130087,"f1":0.32351311414850387,"roc_auc":0.6020699144719784,"pr_auc":0.20533779176339034},
    "random_forest": {"accuracy":0.8510392946809132,"f1":0.2773174704341052,"roc_auc":0.8044953607113304,"pr_auc":0.40246501462675927},
    "gradient_boosting": {"accuracy":0.8563525427687277,"f1":0.26630880489077763,"roc_auc":0.8260155801805384,"pr_auc":0.4483169949856431},
    "xgboost": {"accuracy":0.8548198750510889,"f1":0.2904836638607505,"roc_auc":0.8262898793557036,"pr_auc":0.44171377834099224},
    "linear_svm": {"accuracy":0.8559876218835757,"f1":0.20242522231204527,"roc_auc":0.8261580968770885,"pr_auc":0.44788405460506064},
    "gaussian_naive_bayes": {"accuracy":0.7272873241081333,"f1":0.4400083925306477,"roc_auc":0.7885646728624294,"pr_auc":0.36412201184388493},
}

y_compare = y_test.to_numpy()
paired_predictions = pd.DataFrame({
    "test_row_id": X_test.index.to_numpy(),
    "y_true": y_compare,
})
for name in MODEL_ORDER:
    paired_predictions[f"{name}_score"] = scores[name]
    paired_predictions[f"{name}_pred"] = preds[name]

paired_predictions.to_csv(
    COMPARISON_DIR / "paired_test_predictions.csv.gz",
    index=False,
    compression="gzip",
)

def metric_point(name):
    pred = preds[name]
    score = scores[name]
    tn, fp, fn, tp = confusion_matrix(y_compare, pred, labels=[0,1]).ravel()
    row = {
        "accuracy": accuracy_score(y_compare, pred),
        "precision": precision_score(y_compare, pred, zero_division=0),
        "recall": recall_score(y_compare, pred),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_compare, pred),
        "roc_auc": roc_auc_score(y_compare, score),
        "pr_auc": average_precision_score(y_compare, score),
        "brier_score": np.nan if name == "linear_svm" else brier_score_loss(y_compare, score),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    return row

point = {name: metric_point(name) for name in MODEL_ORDER}

for name in MODEL_ORDER:
    for metric, expected in FROZEN[name].items():
        delta = abs(point[name][metric] - expected)
        if delta > 1e-6:
            raise RuntimeError(
                f"Frozen-result drift: {name} {metric} current={point[name][metric]:.12f} "
                f"expected={expected:.12f} abs_delta={delta:.3g}"
            )

print("Frozen reconciliation passed for all seven models.")


## 4.1 Canonical Model-Comparison Table

The table below combines, for every model:

- literature/model provenance;
- CV mean and 95% fold-based uncertainty;
- held-out test point estimates;
- held-out bootstrap 95% intervals;
- confusion-matrix counts;
- calibration via Brier score where a valid probability exists.

Linear SVM is intentionally left without a Brier score because its uncalibrated decision margin is not a probability.


In [ ]:
comparison_rows = []
metric_names = ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc"]

for name in MODEL_ORDER:
    row = {
        "model_id": name,
        "model": DISPLAY_NAMES[name],
        "model_family": MODEL_FAMILY[name],
        "provenance": PROVENANCE[name],
        "decision_threshold": THRESHOLD[name],
    }

    cv_table = cv_tables[name].set_index("metric")
    test_table = test_ci_tables[name].set_index("metric")

    for metric in metric_names:
        row[f"cv_{metric}_mean"] = float(cv_table.loc[metric, "mean"])
        row[f"cv_{metric}_ci95_lower"] = float(cv_table.loc[metric, "ci95_lower"])
        row[f"cv_{metric}_ci95_upper"] = float(cv_table.loc[metric, "ci95_upper"])

        row[f"test_{metric}"] = float(point[name][metric])
        row[f"test_{metric}_ci95_lower"] = float(test_table.loc[metric, "ci95_lower"])
        row[f"test_{metric}_ci95_upper"] = float(test_table.loc[metric, "ci95_upper"])

    row["test_brier_score"] = point[name]["brier_score"]
    if name != "linear_svm" and "brier_score" in test_table.index:
        row["test_brier_ci95_lower"] = float(test_table.loc["brier_score", "ci95_lower"])
        row["test_brier_ci95_upper"] = float(test_table.loc["brier_score", "ci95_upper"])
    else:
        row["test_brier_ci95_lower"] = np.nan
        row["test_brier_ci95_upper"] = np.nan

    for k in ["tn","fp","fn","tp"]:
        row[k] = int(point[name][k])

    comparison_rows.append(row)

model_comparison = pd.DataFrame(comparison_rows)
model_comparison.to_csv(COMPARISON_DIR / "model_comparison.csv", index=False)

display_cols = [
    "model","test_accuracy","test_precision","test_recall","test_specificity",
    "test_f1","test_roc_auc","test_pr_auc","test_brier_score"
]
display(model_comparison[display_cols].sort_values("test_roc_auc", ascending=False))


## 4.2 Paired Stratified Bootstrap

Every bootstrap replicate resamples positive and negative respondents separately, then applies **the same sampled respondent indices to every model**.

This preserves the paired design. The primary discrimination comparison focuses on the four near-tied models:

- Logistic Regression
- Gradient Boosting
- XGBoost
- Linear SVM

All seven models are retained in the exported pairwise table for transparency. Confidence intervals quantify model-minus-model effect sizes; they are not treated as a license to select a universal winner from a tiny numerical difference.


In [ ]:
PAIRED_BOOT_REPS = 200
PAIRED_BOOT_SEED = 20251010
paired_rng = np.random.default_rng(PAIRED_BOOT_SEED)

pos_compare = np.flatnonzero(y_compare == 1)
neg_compare = np.flatnonzero(y_compare == 0)
paired_metric_names = ["accuracy","precision","recall","specificity","f1","roc_auc","pr_auc"]

boot_records = []
for b in range(PAIRED_BOOT_REPS):
    s_pos = paired_rng.choice(pos_compare, size=len(pos_compare), replace=True)
    s_neg = paired_rng.choice(neg_compare, size=len(neg_compare), replace=True)
    idx = np.concatenate([s_pos, s_neg])
    paired_rng.shuffle(idx)
    yb = y_compare[idx]

    for name in MODEL_ORDER:
        pb = preds[name][idx]
        sb = scores[name][idx]
        tn, fp, fn, tp = confusion_matrix(yb, pb, labels=[0,1]).ravel()
        rec = {
            "replicate": b + 1,
            "model": name,
            "accuracy": accuracy_score(yb, pb),
            "precision": precision_score(yb, pb, zero_division=0),
            "recall": recall_score(yb, pb),
            "specificity": tn / (tn + fp),
            "f1": f1_score(yb, pb),
            "roc_auc": roc_auc_score(yb, sb),
            "pr_auc": average_precision_score(yb, sb),
            "brier_score": np.nan if name == "linear_svm" else brier_score_loss(yb, sb),
        }
        boot_records.append(rec)

paired_boot = pd.DataFrame(boot_records)
paired_boot.to_csv(COMPARISON_DIR / "paired_bootstrap_model_metrics.csv", index=False)

pair_rows = []
for a, b in combinations(MODEL_ORDER, 2):
    a_df = paired_boot[paired_boot.model == a].sort_values("replicate")
    b_df = paired_boot[paired_boot.model == b].sort_values("replicate")

    metrics = paired_metric_names + (
        ["brier_score"] if a != "linear_svm" and b != "linear_svm" else []
    )
    for metric in metrics:
        diff = a_df[metric].to_numpy() - b_df[metric].to_numpy()
        pair_rows.append({
            "model_a": a,
            "model_b": b,
            "metric": metric,
            "point_difference_a_minus_b": point[a][metric] - point[b][metric],
            "bootstrap_mean_difference": float(diff.mean()),
            "ci95_lower": float(np.quantile(diff, 0.025)),
            "ci95_upper": float(np.quantile(diff, 0.975)),
            "ci_excludes_zero": bool(
                np.quantile(diff, 0.025) > 0 or np.quantile(diff, 0.975) < 0
            ),
        })

paired_differences = pd.DataFrame(pair_rows)
paired_differences.to_csv(COMPARISON_DIR / "paired_bootstrap_differences.csv", index=False)

top4 = {"logistic_regression","gradient_boosting","xgboost","linear_svm"}
display(
    paired_differences[
        paired_differences["model_a"].isin(top4)
        & paired_differences["model_b"].isin(top4)
        & paired_differences["metric"].isin(["roc_auc","pr_auc"])
    ]
)


## 4.3 McNemar Tests at the Frozen Decision Boundaries

McNemar's exact test evaluates whether two classifiers have different error rates on the same respondents.

All 21 model pairs are tested. Because this is a family of multiple tests, the notebook applies a **Holm step-down family-wise error correction**.

These tests describe disagreement at each model's frozen/native decision boundary; they do not replace ROC-AUC, PR-AUC, calibration, or the paired bootstrap analysis.


In [ ]:
correct = {name: preds[name] == y_compare for name in MODEL_ORDER}

mcnemar_rows = []
for a, b in combinations(MODEL_ORDER, 2):
    a_correct_b_wrong = int(np.sum(correct[a] & ~correct[b]))
    a_wrong_b_correct = int(np.sum(~correct[a] & correct[b]))
    discordant = a_correct_b_wrong + a_wrong_b_correct

    p = 1.0 if discordant == 0 else float(
        binomtest(
            k=min(a_correct_b_wrong, a_wrong_b_correct),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )

    mcnemar_rows.append({
        "model_a": a,
        "model_b": b,
        "a_correct_b_wrong": a_correct_b_wrong,
        "a_wrong_b_correct": a_wrong_b_correct,
        "discordant_n": discordant,
        "accuracy_difference_a_minus_b": point[a]["accuracy"] - point[b]["accuracy"],
        "p_value_exact": p,
    })

mcnemar = pd.DataFrame(mcnemar_rows).sort_values("p_value_exact").reset_index(drop=True)

m = len(mcnemar)
raw = mcnemar["p_value_exact"].to_numpy()
adjusted = np.maximum.accumulate(np.minimum(1.0, (m - np.arange(m)) * raw))
mcnemar["p_value_holm"] = adjusted
mcnemar["significant_holm_0_05"] = mcnemar["p_value_holm"] < 0.05

mcnemar.to_csv(COMPARISON_DIR / "mcnemar_pairwise_holm.csv", index=False)
display(mcnemar)


## 4.4 Combined ROC and Precision–Recall Curves

These figures use the **same held-out predictions already produced by Models 01–07**. No model is refit and no threshold is tuned.


In [ ]:
plt.figure(figsize=(9,7))
for name in MODEL_ORDER:
    fpr_c, tpr_c, _ = roc_curve(y_compare, scores[name])
    auc = roc_auc_score(y_compare, scores[name])
    plt.plot(fpr_c, tpr_c, label=f"{DISPLAY_NAMES[name]} ({auc:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("BRFSS 2025 — Seven Frozen Models: ROC Comparison")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / "roc_comparison.png", dpi=180)
plt.show()

plt.figure(figsize=(9,7))
for name in MODEL_ORDER:
    p_curve, r_curve, _ = precision_recall_curve(y_compare, scores[name])
    ap = average_precision_score(y_compare, scores[name])
    plt.plot(r_curve, p_curve, label=f"{DISPLAY_NAMES[name]} ({ap:.3f})")
plt.axhline(y=y_compare.mean(), linestyle="--", label=f"Sample prevalence ({y_compare.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("BRFSS 2025 — Seven Frozen Models: Precision–Recall Comparison")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / "pr_comparison.png", dpi=180)
plt.show()


# Part V — Interpretation and Part III Synthesis

The study does **not** define one universal winner from a single metric.

Interpretation is separated into:

- **discrimination:** ROC-AUC and PR-AUC;
- **positive-class retrieval:** recall and F1;
- **false-positive control:** precision and specificity;
- **probability quality:** Brier score for models that emit valid probabilities;
- **paired uncertainty:** bootstrap intervals for model-to-model differences;
- **fixed-boundary disagreement:** Holm-corrected McNemar tests.

The 15.13% diabetes proportion in this modeling cohort remains an **unweighted sample proportion**, not a U.S. population prevalence estimate.


In [ ]:
leaders = {
    "accuracy": model_comparison.loc[model_comparison["test_accuracy"].idxmax(), ["model","test_accuracy"]].to_dict(),
    "precision": model_comparison.loc[model_comparison["test_precision"].idxmax(), ["model","test_precision"]].to_dict(),
    "recall": model_comparison.loc[model_comparison["test_recall"].idxmax(), ["model","test_recall"]].to_dict(),
    "specificity": model_comparison.loc[model_comparison["test_specificity"].idxmax(), ["model","test_specificity"]].to_dict(),
    "f1": model_comparison.loc[model_comparison["test_f1"].idxmax(), ["model","test_f1"]].to_dict(),
    "roc_auc": model_comparison.loc[model_comparison["test_roc_auc"].idxmax(), ["model","test_roc_auc"]].to_dict(),
    "pr_auc": model_comparison.loc[model_comparison["test_pr_auc"].idxmax(), ["model","test_pr_auc"]].to_dict(),
}

probability_models = model_comparison[model_comparison["test_brier_score"].notna()]
best_brier = probability_models.loc[
    probability_models["test_brier_score"].idxmin(),
    ["model","test_brier_score"]
].to_dict()

top4_names = ["logistic_regression","gradient_boosting","xgboost","linear_svm"]
top4_auc = model_comparison[model_comparison.model_id.isin(top4_names)]["test_roc_auc"]
top4_auc_range = float(top4_auc.max() - top4_auc.min())

print("Point-estimate leaders:")
for metric, result in leaders.items():
    print(f"  {metric:12s}: {result['model']} ({result['test_'+metric]:.6f})")
print(f"  brier_score : {best_brier['model']} ({best_brier['test_brier_score']:.6f})")
print()
print(f"ROC-AUC spread among LR / Gradient Boosting / XGBoost / Linear SVM: {top4_auc_range:.6f}")
print("Use paired_bootstrap_differences.csv before interpreting that small spread as meaningful.")

manifest = {
    "study": "BRFSS 2025 Diabetes Classification",
    "canonical_notebook": "BRFSS_2025_Diabetes_Classification.ipynb",
    "single_notebook_pipeline": True,
    "primary_models": MODEL_ORDER,
    "test_n": int(len(y_compare)),
    "paired_bootstrap_reps": PAIRED_BOOT_REPS,
    "paired_bootstrap_seed": PAIRED_BOOT_SEED,
    "mcnemar_multiple_comparison_correction": "Holm FWER",
    "model_retuning_after_test": False,
    "threshold_optimization_on_test": False,
    "linear_svm_brier_score": None,
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
    "xgboost": xgb.__version__,
}
with open(COMPARISON_DIR / "comparison_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
